# SANAD Network RC3 — LensFD Full Dataset (100% coverage)

**Purpose:** A reproducible full-data audit of the published Rice University LensFD measurements, plus verification of the frozen RC3 engine and a strict input-compatibility report.

**No subsampling.** The notebook downloads the **entire ~18-GB** official `data_lensFD.tgz`, enumerates every archive member, hashes every file, and reads **every element of every numeric HDF5 dataset**, with bounded-memory chunks. A partial or corrupt run is marked **INCOMPLETE**, not silently treated as a full test. It writes resumable results to your **private** Google Drive. It never uploads Rice University's raw dataset to GitHub or another public location.

**Important distinction:** LensFD measurements are real. Its self-interference recordings are taken in separate radio-chain runs (up to 32 antennas per run), whereas frozen RC3 requires observation-derived evidence at 300 reporting periods across 7 arrays. The notebook **will not synthesize missing RC3 feature streams** or claim a fault-detection rate from an incompatible reshape. Its `RC3_COMPATIBILITY.json` records the precise gap; the engine and source hashes are checked separately.

**Source and license:** [Rice University RENEW LensFD](https://renew.rice.edu/dataset-lens.html), Zhican Chen et al., *IEEE Transactions on Vehicular Technology* (2023), DOI `10.1109/TVT.2023.3240558`. Rice University grants non-exclusive, non-transferable use for commercial, educational and research purposes; **do not redistribute the original dataset without Rice University's written permission**. Downloading/using it constitutes acceptance of the stated license.

**Resource requirements:** Sufficient Google Drive space for the complete archive (approximately 18 GB), enough Colab temporary disk for the largest individual decompressed member, and time for a full transfer and scan. This is intentionally **not** a 5,000-row or limited-file trial.

In [ ]:
# Install only tools required to inspect the entire real measurement archive.
%pip -q install 'numpy>=1.26,<3' 'h5py>=3.10,<5' 'requests>=2.31,<3' 'scikit-learn>=1.5,<1.9' 'joblib>=1.3,<2' 'pytest>=8,<10'
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, time, requests
ROOT = Path('/content/drive/MyDrive/SANAD_Network/RC3_LensFD_FULL')
RAW = ROOT / 'source_archive_PRIVATE'
RESULTS = ROOT / 'results_full'
RAW.mkdir(parents=True,exist_ok=True)
RESULTS.mkdir(parents=True,exist_ok=True)
SCRATCH = Path('/content')
print('Drive destination:',ROOT)
print('Colab scratch free, GiB:',round(shutil.disk_usage(SCRATCH).free/(2**30),2))
print('Raw LensFD is licensed; keep this Drive folder private.')

## 1. Install the exact full-data runner

The code in the next cell is included **inside this notebook**, so no extra downloads or GitHub branch are needed for source processing. It processes *all* archive files, without `max_records`, random selection, or filename filters.

In [ ]:
from pathlib import Path
Path('/content/lensfd_full_run.py').write_text(r'''"""Full-archive LensFD source audit and frozen RC3 compatibility gate.

Read every byte of every regular archive member and every element in every numeric
HDF5 dataset. No sampling, no silent truncation, no fabricated RC3 features.
Source: Rice University LensFD (Chen et al., IEEE TVT 2023).
License: Rice grants non-exclusive/non-transferable use; raw data must not be redistributed.
"""
from __future__ import annotations
import argparse
import hashlib
import json
import math
import os
import shutil
import tarfile
import tempfile
from datetime import datetime, timezone
from itertools import product
from collections import Counter
from pathlib import Path
from typing import Iterator

import numpy as np

BLOCK_BYTES = 16 * 1024 * 1024
OFFICIAL_URL = 'https://renew.rice.edu/datasets/ashu/traces/LensFD/data_lensFD.tgz'
EXPECTED_FROZEN_COMMIT = '182e77b485cceb770ee6570ba332205c73823c91'
EXPECTED_MODEL_SHA256 = '0a90835bb57a56437fc09211902b581bb22377676f91dce68f309c92e3a95bf4'


def utcnow():
    return datetime.now(timezone.utc).isoformat()


def hash_file(p: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with Path(p).open('rb') as file:
        for b in iter(lambda: file.read(chunk_bytes), b''):
            h.update(b)
    return h.hexdigest()


def atomic_json(path: Path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    with temp.open('w', encoding='utf8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, allow_nan=False)
        f.flush(); os.fsync(f.fileno())
    os.replace(temp, path)


def _json_line(path: Path, obj):
    with Path(path).open('a', encoding='utf8') as f:
        f.write(json.dumps(obj, ensure_ascii=False, allow_nan=False) + '\n')
        f.flush(); os.fsync(f.fileno())


def iter_dataset_slices(shape, itemsize, target_bytes=BLOCK_BYTES) -> Iterator[tuple]:
    """Cover the complete N-D logical HDF5 space exactly once, at bounded memory."""
    if not shape:
        yield ()
        return
    blocks = list(map(int, shape))
    if any(k <= 0 for k in blocks):
        return
    while math.prod(blocks) * max(1, itemsize) > target_bytes:
        largest = max(range(len(blocks)), key=lambda i: blocks[i])
        if blocks[largest] == 1:
            raise MemoryError('Single HDF5 element larger than target block')
        blocks[largest] = (blocks[largest] + 1) // 2
    ranges = [range(0, n, w) for n,w in zip(shape, blocks)]
    for start in product(*ranges):
        yield tuple(slice(s, min(s+b,n)) for s,b,n in zip(start,blocks,shape))


def _scan_numeric(ds) -> dict:
    dtype = ds.dtype
    count = int(math.prod(ds.shape)) if ds.shape else 1
    record = dict(shape=list(ds.shape), dtype=str(dtype), values=count)
    if dtype.kind not in ('b','i','u','f','c'):
        record.update(numeric_analyzed=False, values_scanned=0, reason='Non-plain-numeric HDF5 dtype; complete source bytes still SHA-256 verified')
        return record
    total = 0
    zero = 0
    finite = 0
    abs_sum = 0.0
    sq_sum = 0.0
    mn = float('inf'); mx = float('-inf')
    for sl in iter_dataset_slices(ds.shape, int(dtype.itemsize)):
        vals = np.asarray(ds[sl] if sl else ds[()]).reshape(-1)
        total += len(vals)
        valid = np.isfinite(vals)
        finite += int(np.count_nonzero(valid))
        good = vals[valid]
        if good.size:
            mag = np.abs(good.astype(np.complex128 if np.iscomplexobj(good) else np.float64))
            zero += int(np.count_nonzero(mag == 0))
            abs_sum += float(np.sum(mag, dtype=np.float64))
            sq_sum += float(np.sum(mag * mag, dtype=np.float64))
            mn = min(mn, float(np.min(mag)))
            mx = max(mx, float(np.max(mag)))
    if total != count:
        raise RuntimeError(f'Numeric coverage incomplete {ds.name}: {total}/{count}')
    record.update(numeric_analyzed=True,values_scanned=total,finite_values=finite,
                  nonfinite_values=total-finite,zero_values=zero,
                  mean_abs=abs_sum/finite if finite else None,
                  rms_abs=math.sqrt(sq_sum/finite) if finite else None,
                  min_abs=mn if finite else None,max_abs=mx if finite else None)
    return record


def inspect_hdf5(path: Path):
    import h5py
    rows=[]; n_numeric=0; n_analyzed=0; total_values=0
    with h5py.File(path, 'r') as f:
        def visit(name, item):
            nonlocal n_numeric,n_analyzed,total_values
            if not isinstance(item,h5py.Dataset):
                return
            stats=_scan_numeric(item)
            stats['dataset_path']='/'+name
            stats['attribute_names']=sorted(list(item.attrs.keys()))
            rows.append(stats)
            if stats['numeric_analyzed']:
                n_numeric+=1
                n_analyzed+=stats['values_scanned']
            total_values+=stats['values']
        f.visititems(visit)
        root_attrs=sorted(list(f.attrs.keys()))
    return {'hdf5_datasets':rows,'hdf5_dataset_count':len(rows),
            'numeric_dataset_count':n_numeric,'numeric_values_scanned':n_analyzed,
            'total_logical_values':total_values,'root_attribute_names':root_attrs}


def _safe_member_name(name):
    p=Path(name)
    if p.is_absolute() or '..' in p.parts or '\\' in name or not name or name.startswith('~'):
        raise ValueError('Unsafe archive member path')
    return str(p)


def _load_journal(journal):
    if not journal.exists():return {}
    records={}
    with journal.open(encoding='utf8') as f:
        for line in f:
            if line.strip():
                row=json.loads(line)
                # Append-only retry journal; most recent attempt for each member wins.
                records[row['name']]=row
    return records


def run_full_archive(archive: str | Path, output: str | Path, scratch_dir: str | Path | None = None, strict=True) -> dict:
    archive=Path(archive)
    output=Path(output)
    output.mkdir(parents=True,exist_ok=True)
    if not archive.is_file():raise FileNotFoundError(str(archive))
    arch_size=archive.stat().st_size
    arch_sha=hash_file(archive)  # every byte of the ~18-GB archive is hashed
    lock=output/'SOURCE_LOCK.json'
    if lock.exists():
        previous=json.loads(lock.read_text())
        if (previous['archive_sha256'],previous['archive_size']) != (arch_sha,arch_size):
            raise RuntimeError('Archive has changed; refusing to mix old and new checkpoint outputs')
    else:
        atomic_json(lock, {'official_source':OFFICIAL_URL, 'archive_size':arch_size,
                           'archive_sha256':arch_sha, 'created_utc':utcnow()})
    journal=output/'members.jsonl'
    prior=_load_journal(journal)
    seen=set(); errors=[]; dirs=0; fresh=0
    # stream tar/gzip, one member at a time; no extraction of the full archive
    with tarfile.open(archive,'r|gz') as tar:
        for member in tar:
            name=_safe_member_name(member.name)
            if member.isdir():
                dirs+=1
                continue
            if name in seen: raise RuntimeError(f'Duplicate regular archive member: {name}')
            seen.add(name)
            if name in prior and prior[name]['status']=='OK':
                if prior[name]['size']!=member.size:
                    raise RuntimeError(f'Invalid resume checkpoint: {name}')
                continue
            row={'name':name,'size':member.size,'mtime':member.mtime,'status':'ERROR'}
            tempname=None
            try:
                if not member.isfile():
                    # symbolic links carry no independently supplied data payload
                    row.update(status='ERROR',error='Unsupported non-regular archive object')
                else:
                    free=shutil.disk_usage(str(scratch_dir or tempfile.gettempdir())).free
                    if member.size+1024*1024*200>free:
                        raise OSError(f'Insufficient scratch disk for {name}, need {member.size:,}, free {free:,}')
                    inp=tar.extractfile(member)
                    if inp is None: raise RuntimeError('No member stream')
                    dig=hashlib.sha256()
                    with tempfile.NamedTemporaryFile(prefix='lensfd_',suffix=Path(name).suffix,
                        dir=str(scratch_dir) if scratch_dir else None,delete=False) as temp:
                        tempname=temp.name
                        remaining=member.size
                        while remaining:
                            data=inp.read(min(8*1024*1024,remaining))
                            if not data: raise EOFError(f'Truncated tar member {name}')
                            temp.write(data);dig.update(data);remaining-=len(data)
                    row['sha256']=dig.hexdigest()
                    suffix=Path(name).suffix.lower()
                    if suffix in ('.hdf5','.h5'):
                        row.update(inspect_hdf5(Path(tempname)))
                        row['analysis']='FULL_HDF5_NUMERIC_SCAN'
                    else:
                        row['analysis']='FULL_BYTE_SHA256_ONLY'
                        row['warning']='No numeric HDF5 decode available for this file type'
                    row['status']='OK'
            except Exception as exc:
                row.update(status='ERROR',error=f'{type(exc).__name__}: {exc}')
                errors.append({'name':name,'error':row['error']})
            finally:
                if tempname is not None:
                    try:Path(tempname).unlink(missing_ok=True)
                    except Exception:pass
            _json_line(journal,row)
            prior[name]=row
            fresh+=1
            print(f"[{len(seen)}] {row['status']:5} {name}  bytes={member.size:,}",flush=True)
    regular=[v for v in prior.values() if v['name'] in seen]
    complete=(len(regular)==len(seen) and all(v['status']=='OK' for v in regular))
    source_groups=Counter(Path(r['name']).name.split('-')[0].lower() for r in regular if r['status']=='OK')
    summary={'status':'COMPLETE_FULL_ARCHIVE_AUDIT' if complete else 'INCOMPLETE',
        'official_source':OFFICIAL_URL,'finished_utc':utcnow(),
        'archive_sha256':arch_sha,'archive_bytes_hashed':arch_size,
        'archive_members_regular':len(seen),'archive_members_directories':dirs,
        'source_filename_prefix_counts':dict(sorted(source_groups.items())),
        'members_checked':len(regular),'members_ok':sum(r['status']=='OK' for r in regular),
        'member_bytes_accounted':sum(r['size'] for r in regular if r['status']=='OK'),
        'hdf5_dataset_count':sum(r.get('hdf5_dataset_count',0) for r in regular if r['status']=='OK'),
        'hdf5_numeric_values_scanned':sum(r.get('numeric_values_scanned',0) for r in regular if r['status']=='OK'),
        'hdf5_total_logical_values':sum(r.get('total_logical_values',0) for r in regular if r['status']=='OK'),
        'hdf5_nonnumeric_dataset_count':sum(r.get('hdf5_dataset_count',0)-r.get('numeric_dataset_count',0) for r in regular if r['status']=='OK'),
        'bytes_sampled':0, 'inference_source':'RAW_REAL_RF_MEASUREMENTS',
        'rc3_qualification_on_lensfd':'NOT_YET_TESTED',
        'errors':errors + [{'name':r['name'],'error':r.get('error','unresolved')} for r in regular if r['status']!='OK' and r['name'] not in [e['name'] for e in errors]]}
    atomic_json(output/'FULL_AUDIT_SUMMARY.json',summary)
    if strict and not complete:raise RuntimeError('Full archive audit INCOMPLETE; see FULL_AUDIT_SUMMARY.json')
    return summary


def rc3_contract_report(output_dir:str|Path, frozen_repository_path:str|Path|None=None):
    """Fail closed: LensFD raw IQ is NOT the frozen RC3 300x7 observation trace.

    RC3 requires real upstream evidence streams produced by the frozen Sionna
    feature-generation pathway, not arbitrary tensor reshaping or synthetic rows.
    """
    reasons=[
        'RC3 input is an evidence NPZ of 300 reporting periods x 7 arrays with precomputed v11, fast/strict and temporal delta observations, not raw HDF5 IQ/CSI.',
        'LensFD self-interference recording covers up to 32 antenna ports in any run; seven eight-element RC3 arrays must not be fabricated from asynchronous recordings.',
        'LensFD source does not supply the frozen RC3 temporal feature stream, matching clock, 7-array baselines, or a machine-readable verified fault-label map by default.',
        'Transforming raw IQ/CSI into plausible arrays or random RC3 observation features would not constitute an RC3 evaluation.',
    ]
    out={'dataset':'Rice LensFD','mode':'FROZEN_RC3','decision_threshold':0.50,
         'expected_frozen_commit':EXPECTED_FROZEN_COMMIT,
         'model_sha256':EXPECTED_MODEL_SHA256,'input_contract':'NPZ [2,300,7,3] plus [300,7,3] and v11 [300,7] per stream',
         'adapter_status':'NO_VALIDATED_LENSFD_TO_RC3_FEATURE_ADAPTER',
         'rc3_predictions_produced':0,'rc3_field_fault_detection_rate':None,
         'reasons':reasons,'integrity_check_status':'SEPARATE_FROM_LENSFD_INFERENCE'}
    output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=True)
    atomic_json(output_dir/'RC3_COMPATIBILITY.json',out)
    return out


if __name__=='__main__':
    parser=argparse.ArgumentParser()
    parser.add_argument('--archive',required=True)
    parser.add_argument('--output',required=True)
    parser.add_argument('--scratch',default='/content')
    args=parser.parse_args()
    print(json.dumps(run_full_archive(args.archive,args.output,args.scratch),indent=2))
    print(json.dumps(rc3_contract_report(args.output),indent=2))
''', encoding='utf-8')
import sys
sys.path.insert(0,'/content')
from lensfd_full_run import run_full_archive, rc3_contract_report, hash_file
print('Full-data runner ready. Source bytes:',Path('/content/lensfd_full_run.py').stat().st_size)

## 2. Download **100%** of the official 18-GB archive

The `.part` file is saved in Google Drive. A disconnected download resumes with HTTP Range if the origin supports it, or restarts cleanly without appending a second copy. The archive is not treated as complete until its full expected byte count has arrived. After the download, the next step independently reads and hashes the archive from beginning to end.

In [ ]:
from urllib.parse import urlparse
URL = 'https://renew.rice.edu/datasets/ashu/traces/LensFD/data_lensFD.tgz'
ARCHIVE = RAW/'data_lensFD.tgz'
PART = RAW/'data_lensFD.tgz.part'

def download_entire_archive(url,final,part):
    session=requests.Session()
    with session.head(url,allow_redirects=True,timeout=90) as r:
        advertised=int(r.headers['Content-Length']) if r.ok and r.headers.get('Content-Length','').isdigit() else None
        print('Source advertised total:',f'{advertised:,}' if advertised else 'unknown; will verify during GET')
    if final.exists():
        if advertised is not None and final.stat().st_size!=advertised:
            raise RuntimeError('Existing final archive length disagrees with source. Do not reuse it.')
        print('Existing complete candidate:',final,'bytes=',final.stat().st_size)
        return final
    offset=part.stat().st_size if part.exists() else 0
    headers={'Range':f'bytes={offset}-'} if offset else {}
    with session.get(url,headers=headers,stream=True,timeout=(60,180),allow_redirects=True) as response:
        response.raise_for_status()
        if offset and response.status_code==206:
            cr=response.headers.get('Content-Range','')
            if not cr.startswith(f'bytes {offset}-'):
                raise RuntimeError('Server returned the wrong resume byte range: '+cr)
            expected=int(cr.rsplit('/',1)[1]) if '/' in cr and cr.rsplit('/',1)[1].isdigit() else advertised
            mode='ab'
        elif response.status_code==200:
            if offset:
                print('Server did not support Range resume: restarting archive safely from byte zero.')
            expected=int(response.headers['Content-Length']) if response.headers.get('Content-Length','').isdigit() else advertised
            mode='wb';offset=0
        else:
            raise RuntimeError('Unexpected HTTP code for archive: '+str(response.status_code))
        last_report=offset
        with part.open(mode) as output:
            for chunk in response.iter_content(chunk_size=8*1024*1024):
                if not chunk:continue
                output.write(chunk)
                offset+=len(chunk)
                if offset-last_report>=512*1024*1024:
                    print('Received:',f'{offset/(2**30):.2f} GiB',flush=True)
                    output.flush();os.fsync(output.fileno())
                    last_report=offset
            output.flush();os.fsync(output.fileno())
    total=part.stat().st_size
    if expected is not None and total!=expected:
        raise RuntimeError(f'Incomplete archive: downloaded {total:,} of {expected:,} bytes; rerun this cell to resume')
    if advertised is not None and total!=advertised:
        raise RuntimeError('Archive byte count differs from the publisher HEAD')
    os.replace(part,final)
    print('Full transfer completed:',final, 'bytes=',total)
    return final

ARCHIVE=download_entire_archive(URL,ARCHIVE,PART)

## 3. Full-coverage scan (no sampling)

This operation reads the entire archive, validates gzip/tar structure, writes a SHA-256 for the archive and every file, then computes statistics over **all numeric values** in each HDF5 dataset. It extracts **one HDF5 member at a time** to `/content`, then deletes that temporary copy. Other file formats are fully hashed but not misrepresented as decoded HDF5. If an individual dataset has non-numeric or unsupported compound fields, its raw bytes remain hashed and its decoding status is explicitly recorded. A failed item makes the overall status `INCOMPLETE`.

`members.jsonl` is an append-only checkpoint; when re-run with the same archive, already completed and verified source members are not reprocessed unnecessarily. The archive itself is rehashed before reuse.

In [ ]:
FULL_AUDIT=run_full_archive(ARCHIVE,RESULTS,scratch_dir=SCRATCH,strict=True)
assert FULL_AUDIT['status']=='COMPLETE_FULL_ARCHIVE_AUDIT'
assert FULL_AUDIT['members_ok']==FULL_AUDIT['archive_members_regular']
print(json.dumps(FULL_AUDIT,indent=2))

## 4. Verify frozen RC3 engine, model and original independent Sionna results

This step clones the **exact RC3 frozen commit** and runs the original integrity and prospective-qualification verifiers. It **does not** retrain RC3, alter its 0.50 decision threshold, or present its Sionna performance as LensFD accuracy.

In [ ]:
FROZEN_REF='frozen/rc3-sionna-qualified-2026-11-14'
FROZEN_COMMIT='182e77b485cceb770ee6570ba332205c73823c91'
ENGINE_DIR=Path('/content/SANAD-Network-RC3-FROZEN')
if not ENGINE_DIR.exists():
    subprocess.run(['git','clone','--depth','1','--branch',FROZEN_REF,
        'https://github.com/majdishokri/SANAD-Network-RC3.git',str(ENGINE_DIR)],check=True)
commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ENGINE_DIR,text=True).strip()
assert commit==FROZEN_COMMIT, f'RC3 frozen ref changed: {commit}'
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],cwd=ENGINE_DIR,check=True)
subprocess.run([sys.executable,'verify_freeze.py'],cwd=ENGINE_DIR,check=True)
subprocess.run([sys.executable,'qualification/verify_qualification.py'],cwd=ENGINE_DIR,check=True)
subprocess.run([sys.executable,'-c', 'from rc3 import FrozenRC3; e=FrozenRC3(); assert e.threshold==0.50; print("RC3 model loaded at threshold 0.50")'], cwd=ENGINE_DIR,check=True)
print('Frozen RC3 integrity and model-load check passed at',commit)

## 5. Record measured-domain compatibility and completion

LensFD provides measured radio IQ/CSI, but not the **frozen RC3 observations** (`healthy_feats`/`injected_feats`, V11, temporal delta) produced by the RC3 upstream monitoring pipeline. Also, separate acquisition chains cannot be assumed simultaneous. We therefore **refuse to invent RC3 features**. The audit result is still a complete, documented examination of all published source recordings. See `RC3_COMPATIBILITY.json` for the exact conversion requirements before a valid RC3 inference study can be run.

In [ ]:
COMPATIBILITY=rc3_contract_report(RESULTS,ENGINE_DIR)
final={
    'dataset':'Rice LensFD full 18-GB published archive',
    'source':'https://renew.rice.edu/dataset-lens.html',
    'dataset_full_coverage':FULL_AUDIT['status'],
    'source_archive_sha256':FULL_AUDIT['archive_sha256'],
    'all_archive_regular_files_processed':FULL_AUDIT['members_ok']==FULL_AUDIT['archive_members_regular'],
    'numeric_hdf5_values_scanned':FULL_AUDIT['hdf5_numeric_values_scanned'],
    'frozen_rc3_commit':commit,
    'frozen_rc3_threshold':0.50,
    'rc3_lensfd_adapter_status':COMPATIBILITY['adapter_status'],
    'rc3_fault_detection_rate_on_lensfd':None,
    'research_note':'Real measurements fully processed; no invented model inference features or accuracy claims.'
}
(RESULTS/'RUN_FINAL_STATUS.json').write_text(json.dumps(final,indent=2))
print(json.dumps(final,indent=2))
print('Complete results folder:',RESULTS)
assert final['all_archive_regular_files_processed']

### Files saved to your Google Drive

`SANAD_Network/RC3_LensFD_FULL/source_archive_PRIVATE/data_lensFD.tgz` (complete licensed original), `results_full/SOURCE_LOCK.json`, `results_full/members.jsonl` (every member, HDF5 statistics, SHA-256), `results_full/FULL_AUDIT_SUMMARY.json`, `results_full/RC3_COMPATIBILITY.json`, `results_full/RUN_FINAL_STATUS.json`.

**No subset, no random selection, no imputed timestamps/labels, no fake RC3 accuracy.** This notebook cannot be declared complete until every member has been accounted for.